In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# 1. Cargar AMBOS archivos 
df_intakes = pd.read_csv('./dataset/aac_intakes.csv')
df_outcomes = pd.read_csv('./dataset/aac_outcomes.csv')

# 2. Renombrar columnas de 'outcomes'
df_outcomes = df_outcomes.rename(columns={
    'datetime': 'fecha_hora_salida',
    'outcome_type': 'tipo_resultado'
})

# 3. Renombrar columnas de 'intakes'
df_intakes = df_intakes.rename(columns={
    'datetime': 'fecha_hora_entrada',
    'intake_type': 'tipo_entrada',
    'intake_condition': 'condicion_entrada'
})

# 4. Importante: ambas tablas tienen la columna 'animal_id' que las relaciona
# Usamos un 'inner join' para quedarnos solo con animales que tienen tanto entrada como salida
df_completo = pd.merge(df_intakes, df_outcomes, on='animal_id')

print(f"Filas en el nuevo dataset unido: {len(df_completo)}")
df_completo.head()

Filas en el nuevo dataset unido: 100230


,age_upon_intake,animal_id,animal_type_x,breed_x,color_x,fecha_hora_entrada,datetime2,found_location,condicion_entrada,tipo_entrada,...,animal_type_y,breed_y,color_y,date_of_birth,fecha_hora_salida,monthyear,name_y,outcome_subtype,tipo_resultado,sex_upon_outcome
0,8 years,A706918,Dog,English Springer Spaniel,White/Liver,2015-07-05T12:59:00.000,2015-07-05T12:59:00.000,9409 Bluegrass Dr in Austin (TX),Normal,Stray,...,Dog,English Springer Spaniel,White/Liver,2007-07-05T00:00:00,2015-07-05T15:13:00,2015-07-05T15:13:00,Belle,NaN,Return to Owner,Spayed Female
1,11 months,A724273,Dog,Basenji Mix,Sable/White,2016-04-14T18:43:00.000,2016-04-14T18:43:00.000,2818 Palomino Trail in Austin (TX),Normal,Stray,...,Dog,Basenji Mix,Sable/White,2015-04-17T00:00:00,2016-04-21T17:17:00,2016-04-21T17:17:00,Runster,NaN,Return to Owner,Neutered Male
2,4 weeks,A665644,Cat,Domestic Shorthair Mix,Calico,2013-10-21T07:59:00.000,2013-10-21T07:59:00.000,Austin (TX),Sick,Stray,...,Cat,Domestic Shorthair Mix,Calico,2013-09-21T00:00:00,2013-10-21T11:39:00,2013-10-21T11:39:00,NaN,Partner,Transfer,Intact Female
3,4 years,A682524,Dog,Doberman Pinsch/Australian Cattle Dog,Tan/Gray,2014-06-29T10:38:00.000,2014-06-29T10:38:00.000,800 Grove Blvd in Austin (TX),Normal,Stray,...,Dog,Doberman Pinsch/Australian Cattle Dog,Tan/Gray,2010-06-29T00:00:00,2014-07-02T14:16:00,2014-07-02T14:16:00,Rio,NaN,Return to Owner,Neutered Male
4,2 years,A743852,Dog,Labrador Retriever Mix,Chocolate,2017-02-18T12:46:00.000,2017-02-18T12:46:00.000,Austin (TX),Normal,Owner Surrender,...,Dog,Labrador Retriever Mix,Chocolate,2015-02-18T00:00:00,2017-02-21T17:44:00,2017-02-21T17:44:00,Odin,NaN,Return to Owner,Neutered Male


In [ ]:
# 5. Filtrar solo los 4 outcomes principales
outcomes_principales = ['Adoption', 'Transfer', 'Return to Owner', 'Euthanasia']
df_final = df_completo[df_completo['tipo_resultado'].isin(outcomes_principales)].copy()

# 6. se crea la variable objetivo 'y', esta vez serán 4 clases
le = LabelEncoder()
y = le.fit_transform(df_final['tipo_resultado'])
labels = le.classes_
print(f"Clases a predecir: {labels}")

Clases a predecir: ['Adoption' 'Euthanasia' 'Return to Owner' 'Transfer']


In [ ]:
# 7. Crear un DataFrame X vacío
X = pd.DataFrame()

# nuevas características
X["tipo_entrada"] = df_final["tipo_entrada"]
X["condicion_entrada"] = df_final["condicion_entrada"]

X["tiene_nombre"] = (
    df_final["name_x"].notnull().astype(int)
)  # 'name_x' es el nombre de 'intakes'

# Se usa la info de 'salida', es más relevante
X["esterilizado"] = df_final["sex_upon_outcome"].apply(
    lambda x: (
        "Si"
        if "Neutered" in str(x) or "Spayed" in str(x)
        else ("No" if "Intact" in str(x) else "Desconocido")
    )
)

X["sexo"] = df_final["sex_upon_outcome"].apply(
    lambda x: (
        "Macho"
        if "Male" in str(x)
        else ("Hembra" if "Female" in str(x) else "Desconocido")
    )
)

X["tipo_animal"] = df_final["animal_type_x"]  # 'animal_type_x'

X["es_mix"] = df_final["breed_x"].apply(
    lambda x: 1 if "Mix" in str(x) or "/" in str(x) else 0
)

In [4]:
# 8. Aplicar One-Hot Encoding
# 'dias_en_refugio', 'tiene_nombre', 'es_mix' ya son numéricas
columnas_categoricas = ['tipo_entrada', 'condicion_entrada', 'esterilizado', 'sexo', 'tipo_animal']
X_encoded = pd.get_dummies(X, columns=columnas_categoricas)

# 9. Dividir los datos
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# 10. Entrenar el RandomForest
modelo_final = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
modelo_final.fit(X_train, y_train)

# 11. Evaluar el modelo
y_predicciones = modelo_final.predict(X_test)
accuracy = accuracy_score(y_test, y_predicciones)

print(f"\n--- RESULTADO CON DATOS UNIDOS (INTAKES + OUTCOMES) ---")
print(f"Precisión Total: {accuracy * 100:.2f}%")
print("Reporte de Clasificación:")
print(
    classification_report(
        y_test, y_predicciones, target_names=labels
    )
)


--- RESULTADO CON DATOS UNIDOS (INTAKES + OUTCOMES) ---
Precisión Total: 62.39%
Reporte de Clasificación:
                 precision    recall  f1-score   support

       Adoption       0.74      0.55      0.63      8741
     Euthanasia       0.68      0.71      0.69      1315
Return to Owner       0.47      0.86      0.61      4541
       Transfer       0.75      0.52      0.61      5171

       accuracy                           0.62     19768
      macro avg       0.66      0.66      0.64     19768
   weighted avg       0.68      0.62      0.63     19768

